# Program for training simple model for back-classification

Create kernel: `uv run ipython kernel install --user --name="python-classification"`
Switch to this kernel (see top right in jupyter)

Also ran one time: `mlflow experiments create --experiment-name "back_classification" --artifact-location s3://projet-aiml4os-wp10/Cluster5/mlflow_artifacts`. Not sure if I need to now as have changed things around

The program is divided into the following general processes:

- [Initial setup](#setup)
- [Read in data](#read)
- [Filter data](#filter)
- [Pre-process text variables](#preprocess)
- [Train model](#train)

<a id="setup"></a>
## Initial Setup
Here we fetch the necessary Python packages for the program. This also includes fetching additional functions from the utils folder and variables from the configuration file (config).

In [1]:
import sys
print(sys.executable)

/home/onyxia/work/WP10-Cluster5-backclassification/.venv/bin/python


In [2]:
import pandas as pd
import os

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.calibration import CalibratedClassifierCV

import nltk
from nltk.corpus import stopwords

from utils.one_to_one import get_1to1_dict, get_problem_NACE
from utils.rules_classification import norwegian_rules
from utils.text_processing import preprocess
from config import path_info, bucket_info, model_hyperparameter

import mlflow
import mlflow.sklearn

In [3]:
# Location for model and data
bucket_url = bucket_info["bucket_url"]
bucket = "projet-aiml4os-wp10/Cluster5"

train_path = f"{bucket_url}/train_doublenace_2026-03-23.parquet"
conversion_path = f"{bucket_url}/NACE2.1-NACE2_Table_V1.05.xlsx"

In [13]:
os.environ["S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["AWS_S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]

In [14]:
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'

In [17]:
os.getenv('AWS_S3_ENDPOINT')

'https://minio.lab.sspcloud.fr'

In [11]:
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY: ")
os.environ["AWS_SESSION_TOKEN"] = getpass("AWS_SESSION_TOKEN: ")


AWS_ACCESS_KEY_ID:  ········
AWS_SECRET_ACCESS_KEY:  ········
AWS_SESSION_TOKEN:  ········


In [ ]:
# In bash, $ export xxx = yyy  - but doesn't import to correct env

In [ ]:
# Original experiment setup # 

In [20]:
# Set MLflow experiment for ML model if not set from before

if not mlflow.get_experiment_by_name("back_classification"):
    mlflow.create_experiment(
        "back_classification",
        artifact_location=f"s3://{bucket}/mlflow_artifacts/"
    )

mlflow.set_experiment("back_classification")

<Experiment: artifact_location='s3://projet-aiml4os-wp10/Cluster5/mlflow_artifacts/', creation_time=1791899027423, experiment_id='2', last_update_time=1791899027423, lifecycle_stage='active', name='back_classification', tags={}, workspace='default'>

<a id="read"></a>
## Read in data

In [21]:
# Read in training data
train = pd.read_parquet(path_info["train_path"])
print(f'Number of observations in training: {train.shape[0]}')

Number of observations in training: 440746


<a id="filter"></a>
## Filter data
**Remove 1-to-1 groups**

In [22]:
one_to_one_dict = get_1to1_dict(path_info["conversion_path"])
mask_1 = train.nace_21_code.isin(one_to_one_dict.keys())
print(f'Number of observations in one-to-one nace: {mask_1.sum()}')

train = train.loc[~mask_1, :].copy()

Number of observations in one-to-one nace: 347600


**Remove problem groups**

In [23]:
problem_nace = get_problem_NACE(path_info["conversion_path"])
mask_problem = train.nace_21_code.isin(problem_nace)
print(f'Number of observations in problem nace: {mask_problem.sum()}')

train = train.loc[~mask_problem, :].copy()

Number of observations in problem nace: 2


**Remove rules groups**

In [24]:
mask_rules = train.nace_21_code.isin(norwegian_rules.keys())
print(f'Number of observations in rules nace: {mask_rules.sum()}')

train = train.loc[~mask_rules, :].copy()

Number of observations in rules nace: 27


**Remove groups with too small counts (<3)**

In [25]:
# Check minimum number
tab = train.groupby("nace_20_code").orgnr.count()
print(f'Minium number of observations in NACE class: {min(tab)}')

Minium number of observations in NACE class: 1


In [26]:
small_groups = tab[tab<3].index
mask_small = train.nace_20_code.isin(small_groups.tolist())
print(f"The following NACE rev. 2.0 have less than 3 observations and have been removed: {small_groups.tolist()}")
print(f"Number of observations removed due to less than 3 observations: {mask_small.sum()}")

train = train.loc[~mask_small,:].copy()

The following NACE rev. 2.0 have less than 3 observations and have been removed: ['16.21', '16.23', '33.16', '35.14', '35.23']
Number of observations removed due to less than 3 observations: 8


<a id="preprocess"></a>
## Preprocess text

In [27]:
# Get stopwords
nltk.download('stopwords') # one time to use locally
norwegian_stopwords = set(stopwords.words('norwegian'))|{"as"}

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [28]:
# Create text variable
train["company_name"] = train["company_name"].fillna("")
train["company_activity"] = train["company_activity"].fillna("")
train["company_text"] = train["company_name"] + " " + train["company_activity"]

In [29]:
# Preprocess and add in nace 2.1
train["company_text_processed"] = preprocess(train, 
                                             text_variable="company_text", 
                                             stopwords = norwegian_stopwords, 
                                             nace_21_variable = "nace_21_code"
                                            )

<a id="train"></a>
## Train model


In [30]:
# using small set for testing
mask_little = train.nace_20_code.isin(["18.12","25.61","28.21","31.02"])
train = train.loc[mask_little,:].copy()
train.groupby("nace_20_code").count()

,orgnr,company_name,company_activity,company_purpose,language,number_of_employees,orgform,orgform_name_nb,orgform_name_en,date_of_incorporation,website,sector_code,sector_name_nb,sector_name_en,nace_21_code,nace_21_name_nb,nace_21_name_en,company_text,company_text_processed
nace_20_code,,,,,,,,,,,,,,,,,,,
18.12,169,169,169,127,169,37,169,169,169,127,27,169,169,169,169,169,169,169,169
25.61,467,467,467,352,467,131,467,467,467,352,58,463,463,463,467,467,467,467,467
28.21,18,18,18,18,18,8,18,18,18,18,4,18,18,18,18,18,18,18,18
31.02,131,131,131,87,131,24,131,131,131,87,18,131,131,131,131,131,131,131,131


In [31]:
with mlflow.start_run() as run:
    pipe_svc = Pipeline([('countvec', CountVectorizer(max_features=model_hyperparameter['max_features'])),
                             ('tf', TfidfTransformer(use_idf=True)),
                             ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=model_hyperparameter['C']),
                                                            cv=3, 
                                                            method='sigmoid',
                                                            n_jobs=5),
                             )
                            ])
    pipe_svc.fit(train.company_text_processed , train.nace_20_code)

    result = mlflow.sklearn.log_model(sk_model=pipe_svc, name="model")
    print("Model URI:", result.model_uri)

2026/10/13 14:21:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/13 14:21:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Model URI: models:/m-6dd8a7b6c4454de19e8eb65688f6e2b2


In [32]:
pipe_svc

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('countvec', ...), ('tf', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


### MISC CODE

In [27]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

print("MLflow version:", mlflow.__version__)

experiments = client.search_experiments(
    view_type=mlflow.entities.ViewType.ALL
)

for exp in experiments:
    if exp.name == "back_classification":
        print(
            "ID:", exp.experiment_id,
            "| Status:", exp.lifecycle_stage,
            "| Artifact location:", exp.artifact_location
        )

MLflow version: 3.10.1
ID: 1 | Status: deleted | Artifact location: s3://projet-aiml4os-wp10/Cluster5/mlflow_artifacts/


In [28]:
client.restore_experiment("1")

# Rename it to free up the original name
client.rename_experiment(
    "1",
    "back_classification_old"
)


In [ ]:
# Fixes, rename old experiment and start with fresh ID

In [19]:

if not mlflow.get_experiment_by_name("back_classification"):
    new_id = client.create_experiment(
        "back_classification",
        artifact_location=f"s3://{bucket}/mlflow_artifacts/"
    )

mlflow.set_experiment("back_classification")

print(new_id)
print(client.get_experiment(new_id).artifact_location)

2


NameError: name 'client' is not defined